# Cross-Validation — how much should you trust one split?

## Why this notebook exists

So far every score in this lesson came from **one** `train_test_split`. That split is random.
Change the random seed and you get a different train set, a different test set, and a
different score.

So when notebook 1 told you the 8-degree model had a test RMSE of *X*, a fair question is:
how much of that number was the model, and how much was luck?

This notebook answers that, and gives you the two diagnostic plots — the **validation curve**
and the **learning curve** — that turn the bias-variance story into something you can read off
a chart.

> **Where this sits.** Notebook 1 gave you train/test split and the complexity curve.
> This notebook makes the *estimate itself* trustworthy. The **Hyperparameter Tuning** lesson
> uses it to pick settings; the **Advanced ML-Pipelines** lesson puts it inside a pipeline
> where it is safe from leakage.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.model_selection import (
    KFold,
    LearningCurveDisplay,
    ShuffleSplit,
    ValidationCurveDisplay,
    cross_val_score,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RSEED = 42

## The data

The same white wine data used in the regularization notebook — 4,898 wines, eleven physico-
chemical measurements, and a quality score to predict.

In [ ]:
df = pd.read_csv("data/winequality-white.csv", sep=";")

X = df.drop(columns="quality")
y = df["quality"]

print(X.shape, y.shape)
df.head(2)

## One split is a coin flip

Fit the same model five times, changing nothing but the split's random seed.

In [ ]:
model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))

scores = []
for seed in range(5):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=seed
    )
    model.fit(X_train, y_train)
    scores.append(model.score(X_test, y_test))
    print(f"random_state={seed}   test R² = {scores[-1]:.4f}")

print(f"\nspread across seeds: {max(scores) - min(scores):.4f}")

Same data, same model, same code — and the score moves in the third decimal place or worse.

If you compared two models and they differed by less than that spread, you learned nothing.
**A single split cannot tell you whether a difference is real.**

## K-fold cross-validation

Instead of one split, make `k` of them systematically. Chop the data into `k` equal folds.
Hold out fold 1, train on the rest, score. Hold out fold 2, train on the rest, score. And so
on, until every row has been in the test set exactly once.

```
fold:   1     2     3     4     5
      [test][ train train train train ]   → score 1
      [train][test][ train train train ]  → score 2
      [train train][test][ train train ]  → score 3
      [train train train][test][ train ]  → score 4
      [train train train train][test]     → score 5
```

You end up with `k` scores instead of one. Their **mean** is a better estimate, and their
**standard deviation** tells you how much the estimate wobbles.

In [ ]:
cv_scores = cross_val_score(model, X, y, cv=5, scoring="r2")

print("fold scores:", np.round(cv_scores, 4))
print(f"mean R²    : {cv_scores.mean():.4f}")
print(f"std        : {cv_scores.std():.4f}")

**Always report both numbers.** `0.28 ± 0.02` is a claim you can defend. `0.28` on its own
is not — it hides whether the next split would have said 0.27 or 0.19.

A useful habit: quote the result as an interval.

In [ ]:
print(f"R² = {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

## `cross_validate` — more than one number at a time

`cross_val_score` returns test scores only. `cross_validate` gives you several metrics at
once, and — crucially for this lesson — the **training** score alongside the test score.

In [ ]:
results = cross_validate(
    model,
    X,
    y,
    cv=5,
    scoring=["r2", "neg_root_mean_squared_error"],
    return_train_score=True,
)

summary = pd.DataFrame(results)
summary[["train_r2", "test_r2",
         "train_neg_root_mean_squared_error", "test_neg_root_mean_squared_error"]].round(4)

`return_train_score=True` is what turns cross-validation into a bias-variance diagnostic.

- **train score ≈ test score, both low** → underfitting. The model is too rigid; more data
  will not help.
- **train score ≫ test score** → overfitting. The model memorised the training folds.

Read the gap directly:

In [ ]:
gap = results["train_r2"].mean() - results["test_r2"].mean()
print(f"mean train R² : {results['train_r2'].mean():.4f}")
print(f"mean test  R² : {results['test_r2'].mean():.4f}")
print(f"gap           : {gap:.4f}")

### Choosing how to split

`cv=5` uses `KFold` under the hood for a regression task. You can pass a splitter object
instead when you want control over shuffling or the number of repeats.

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=RSEED)
shuffle = ShuffleSplit(n_splits=5, test_size=0.25, random_state=RSEED)

for name, cv in [("KFold (shuffled)", kfold), ("ShuffleSplit", shuffle)]:
    s = cross_val_score(model, X, y, cv=cv, scoring="r2")
    print(f"{name:20s} R² = {s.mean():.4f} ± {s.std():.4f}")

Look carefully at those two lines against the plain `cv=5` result earlier
(`0.246 ± 0.058`). Shuffling cut the standard deviation to roughly a third.

That is not noise — it is a warning. Plain `cv=5` slices the file **in the order the rows
happen to sit on disk**, and this file is not in random order. Each fold therefore gets a
systematically different slice of wines, and the fold-to-fold spread reflects that ordering
as much as it reflects the model.

**Shuffle unless you have a reason not to.** The reasons not to are real and you will meet
them in the Advanced ML-Pipelines lesson: when rows are grouped by some source, or ordered in
time, shuffling is exactly the wrong move because it leaks information between folds.

`KFold` guarantees every row is tested exactly once. `ShuffleSplit` draws a fresh random
split each time, so rows can repeat across folds — useful when you want many estimates from a
large dataset without the fold count dictating the test-set size.

## The validation curve

Notebook 1 plotted error against polynomial degree by hand. That plot has a name — a
**validation curve** — and scikit-learn will build it for you, with cross-validation at every
point instead of a single split.

Here we sweep Ridge's `alpha`, the regularisation strength from the previous notebook.

In [ ]:
alphas = np.logspace(-2, 4, 13)

display = ValidationCurveDisplay.from_estimator(
    make_pipeline(StandardScaler(), Ridge()),
    X,
    y,
    param_name="ridge__alpha",
    param_range=alphas,
    cv=5,
    scoring="r2",
    score_type="both",
    n_jobs=-1,
)
display.ax_.set_xscale("log")
display.ax_.set_xlabel("alpha (regularisation strength)")
display.ax_.set_title("Validation curve — Ridge on wine quality")
plt.show()

Read it left to right:

- **Left (small alpha)** — barely any regularisation. Train and test scores sit close together
  here because this model is not complex enough to overfit badly, but note the shape.
- **Right (large alpha)** — the penalty crushes the coefficients towards zero. Both curves
  fall together: the model has been regularised into **underfitting**.
- **The best alpha** is wherever the *test* curve peaks — not where the train curve is best,
  which is always at the far left.

The shaded bands are the standard deviation across folds. When two settings' bands overlap,
you cannot claim one is better.

## The learning curve

A different question: forget the hyperparameters — **would more data help?**

A learning curve plots the score against the number of training examples.

In [ ]:
display = LearningCurveDisplay.from_estimator(
    model,
    X,
    y,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=5,
    scoring="r2",
    score_type="both",
    n_jobs=-1,
)
display.ax_.set_xlabel("number of training examples")
display.ax_.set_title("Learning curve — Ridge on wine quality")
plt.show()

What this particular curve shows: with 391 training rows the model scores R² = 0.15 on
held-out data against 0.31 on its own training rows — a gap of 0.16. By 3,918 rows the gap has
closed to 0.04 (0.246 against 0.284). The test curve is **still creeping upward** at the
right-hand edge, but the slope has almost flattened.

Read that as: more data has clearly helped, and a little more would help a little more — but
the returns have mostly been collected. The remaining gap between 0.28 and the perfect 1.0 is
not a data problem, it is a *model* problem. A linear model on eleven features cannot capture
what determines wine quality, and no quantity of extra rows will change that.

Two shapes worth recognising:

- **The curves have converged and flattened** (what you see here) — the model has learned
  everything it can from this data. Collecting more rows will not help; you need a more
  flexible model or better features.
- **The test curve is still climbing at the right-hand edge** — the model is data-starved.
  More rows are the cheapest available improvement.

This is the plot that answers "should we spend money on more data?" — and it is the reason
the question has an evidence-based answer rather than an opinion.

## Summary

- One `train_test_split` gives one noisy estimate. Changing the seed changed our R² by
  enough to swamp a real model difference.
- **K-fold cross-validation** averages over `k` splits. Report the **mean and the standard
  deviation** — a score without a spread is not a claim.
- `cross_val_score` gives test scores; **`cross_validate`** gives several metrics and, with
  `return_train_score=True`, the train/test gap that diagnoses over- and underfitting.
- The **validation curve** shows score against one hyperparameter — where the *test* curve
  peaks is the setting to use.
- The **learning curve** shows score against training-set size — it tells you whether more
  data would help.

> **Where this goes next.** The **Decision Trees** and **Random Forest** lessons use
> cross-validation to compare models. The **Hyperparameter Tuning** lesson automates the
> validation-curve search across many parameters at once. The **Advanced ML-Pipelines** lesson
> shows why the scaler had to be inside a `Pipeline` in every cell above — fit it outside the
> cross-validation loop and you leak test information into training.